# 04 — Parallel Agents: Fan-Out and Fan-In

**Mode:** Offline  
**Level:** Fundamentals  
**Estimated time:** 30 minutes

[Watch the original lesson video](https://www.youtube.com/watch?v=VNZ_lhljPBc)

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A fan-out/fan-in analysis workflow. Three specialists process the same input concurrently, a combiner tracks completion by session, and one final Spore contains the aggregate. A controlled partial failure shows how policy changes the result.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(4, 'Parallel Agents: Fan-Out and Fan-In')


## Prerequisites and setup

**Course prerequisites:** Complete `course-02-research-pipeline`.

**Execution requirements:** Offline. No API key or external service is required.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Fan one Spore out to multiple agents.
- Track completion by correlation ID.
- Aggregate results exactly once under concurrency.
- Represent partial failure explicitly.


## Mental model

**Fan-out** means several subscribers independently receive the same input. **Fan-in** means a combiner waits for their correlated results. Since handlers can finish in any order, aggregation state needs a lock and an exactly-once guard. Partial failure is a product decision: fail the whole request, return a degraded result, or retry a specialist.


In [ ]:
show_flow(
('Input', 'session-42', 'spore'),
('Specialists', 'parallel fan-out', 'agent'),
('Combiner', 'join by session', 'reef'),
('Final', 'one aggregate', 'spore'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Prepare correlated aggregation state

A lock protects shared buckets because specialist handlers run concurrently. The `combined_sessions` set prevents duplicate final emission.


In [ ]:
import threading
import time

from praval import agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef

reset_reef()
session_results = {}
combined_sessions = set()
final_results = []
result_spores = []
result_lock = threading.Lock()
expected = {"sentiment", "keywords", "length"}


### What just happened?

The aggregation policy now names all required specialists and protects its mutable state.

### Why this matters

Concurrent correctness should be explicit before handlers begin writing results.


### Define three specialists

Each handler sleeps for a different small duration to make completion order visible. The delay represents independent work, not synchronization.


In [ ]:
def emit_analysis(spore, analyzer, result, delay):
    time.sleep(delay)
    stage(analyzer, "analysis complete", str(result), spore)
    broadcast({
        "type": "analysis_result",
        "session_id": spore.knowledge["session_id"],
        "analyzer": analyzer, "result": result,
    })


@agent("course-sentiment", responds_to=["analyze"], auto_broadcast=False)
def sentiment(spore):
    emit_analysis(spore, "sentiment", "positive", 0.08)


@agent("course-keywords", responds_to=["analyze"], auto_broadcast=False)
def keywords(spore):
    words = [word.strip(".,!") for word in spore.knowledge["text"].split()]
    emit_analysis(spore, "keywords", words[-3:], 0.03)


@agent("course-length", responds_to=["analyze"], auto_broadcast=False)
def length(spore):
    emit_analysis(spore, "length", len(spore.knowledge["text"].split()), 0.05)


### What just happened?

All three agents subscribe to `analyze`, so one input will schedule all three. Each response carries the same session ID.

### Why this matters

Specialists remain independently replaceable because they share only a message contract.


### Define the exactly-once combiner

The combiner adds each result to its session bucket. It publishes only when the expected specialist set is complete and the session has not been emitted.


In [ ]:
@agent("course-combiner", responds_to=["analysis_result"], auto_broadcast=False)
def combiner(spore):
    result_spores.append(spore)
    session = spore.knowledge["session_id"]
    with result_lock:
        bucket = session_results.setdefault(session, {})
        bucket[spore.knowledge["analyzer"]] = spore.knowledge["result"]
        ready = set(bucket) == expected and session not in combined_sessions
        if ready:
            combined_sessions.add(session)
            combined = dict(bucket)
    stage("combiner", "result received", spore.knowledge["analyzer"], spore)
    if ready:
        broadcast({
            "type": "combined_analysis", "session_id": session,
            "analysis": combined, "partial": False,
        })


@agent("course-parallel-output", responds_to=["combined_analysis"], auto_broadcast=False)
def final_output(spore):
    final_results.append(spore.knowledge)
    stage("output", "fan-in complete", spore.knowledge["session_id"], spore)


### What just happened?

Aggregation is order-independent. The lock guards the check-and-mark operation so two nearly simultaneous results cannot emit two final messages.

### Why this matters

Exactly-once emission is a property of the combiner policy, not something the Reef can infer from arbitrary application state.


### Run the fan-out/fan-in workflow

The initial input contains a stable session ID. The assertions prove one final result contains every specialist output.


In [ ]:
start_agents(
    sentiment, keywords, length, combiner, final_output,
    initial_data={
        "type": "analyze", "session_id": "session-42",
        "text": "Praval makes concurrent agent behavior visible and composable.",
    },
    channel="course-parallel",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=10)
assert len(final_results) == 1
assert set(final_results[0]["analysis"]) == expected


### What just happened?

The keywords handler finished first, but the combiner waited for the full set. Only one terminal result was emitted.

### Why this matters

Completion order should not change correctness.


In [ ]:
show_json(final_results[0], "Joined result")
show_spore(result_spores[0], "One specialist result Spore")
show_timeline()


### Make partial failure visible

A degraded aggregation should say which specialist is missing. This small policy example does not alter the successful run above.


In [ ]:
def partial_result(session_id, bucket, required):
    missing = sorted(required - set(bucket))
    return {
        "type": "combined_analysis",
        "session_id": session_id,
        "analysis": dict(bucket),
        "partial": bool(missing),
        "missing": missing,
    }


degraded = partial_result(
    "session-43", {"sentiment": "positive", "length": 8}, expected
)
assert degraded["partial"] and degraded["missing"] == ["keywords"]
show_json(degraded, "Explicit partial result")


### What just happened?

The result is usable but cannot be confused with a complete aggregate.

### Why this matters

Explicit degradation lets downstream agents decide whether to accept, retry, or reject incomplete work.


## Your turn

Run two session IDs through the same specialists and prove their buckets never mix. Then choose a policy for one missing specialist.


In [ ]:
# Use session-44 and session-45 in separate input Spores.
# Assert set(session_results) includes both IDs and each bucket is independent.
# Decide: retry, emit partial_result(...), or broadcast a terminal error Spore.


## Common mistake

**Mistake:** Using `len(bucket) == 3` without checking which three specialist names arrived.

**Correction:** Compare the actual keys to the expected set; duplicate or unknown producers must not satisfy completion.


<details>
<summary>Under the hood</summary>

The Reef schedules subscribers independently, so application state can be updated from multiple worker threads. A lock must cover both readiness testing and the exactly-once marker. Copy the completed bucket before releasing the lock.

</details>


## Recap

- Fan-out uses multiple subscribers to one message type.
- Fan-in requires correlation, completion tracking, and exactly-once emission.
- Result order must not affect aggregate correctness.
- Partial failure needs an explicit downstream-visible policy.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
for function in (sentiment, keywords, length, combiner, final_output):
    function._praval_agent.close()
assert reef.shutdown(timeout=10)


### Next lesson

Continue to `05_tool_use.ipynb` to give agents validated capabilities through JSON-schema tool contracts.
